In [1]:
import json
import ollama

# Escolha o modelo que você baixou no Ollama (ex: llama3.1, qwen2.5)
MODELO_LOCAL = 'llama3'
# Bert

def gerar_issue(document_text, max_tentativas=3):
    prompt = f"""
    Com base na seguinte descrição de um sistema de software, crie as issues de projeto necessárias.
    As issues geradas devem abranger todo o ciclo de vida do projeto, desde o seu início absoluto (criação do projeto do zero, configurações de repositório, infraestrutura) até a sua entrega final (testes, CI/CD e deploy em produção).
    Retorne APENAS um objeto JSON válido. Este objeto deve conter uma única chave chamada "issues", que será uma lista de objetos.
    Cada objeto dentro dessa lista representará uma issue e deve conter exatamente estas chaves:
    "issue_title" (string), "issue_type" (com o valor exato "Feature") e "description" (string curta).

    Descrição: {document_text}
    """

    for tentativa in range(max_tentativas):
        try:
            resposta = ollama.generate(
                model=MODELO_LOCAL,
                prompt=prompt,
                format='json'
            )

            texto_limpo = resposta['response'].strip()
            return json.loads(texto_limpo)

        except json.JSONDecodeError as e:
            print(f"  [!] Erro ao interpretar o JSON gerado (Tentativa {tentativa + 1}/{max_tentativas}): {e}")
        except Exception as e:
            print(f"  [!] Erro de comunicação com o Ollama: {e}")
            break

    return {"erro": "Falha ao gerar issue após várias tentativas"}

# 1. Lê o arquivo original
print("Carregando Dataset_dirty.json...")
try:
    with open('Dataset_dirty.json', 'r', encoding='utf-8') as f:
        dataset = json.load(f)
except FileNotFoundError:
    print("Erro: O arquivo Dataset_dirty.json não foi encontrado no diretório.")
    exit(1)

# Pegamos apenas os 10 primeiros itens (pode remover o "[:10]" se quiser rodar tudo depois)
dataset_reduzido = dataset

# Lista que vai guardar os nossos novos documentos limpos
novo_dataset_limpo = []

# 2. Percorre os inputs e gera o formato desejado (input/output)
for i, item in enumerate(dataset_reduzido):
    doc_text = item.get('document', '')

    if doc_text:
        print(f"[{i+1}/{len(dataset_reduzido)}] Analisando texto e gerando issue...")

        # Chama a IA para gerar o output
        issue_gerada = gerar_issue(doc_text)

        # Cria um documento NOVO contendo apenas o input e o output
        novo_documento = {
            "input": doc_text,
            "output": issue_gerada
        }

        # Adiciona na nossa lista limpa
        novo_dataset_limpo.append(novo_documento)

# 3. Salva o resultado no arquivo final
print("\nSalvando os resultados...")
with open('Dataset_clean.json', 'w', encoding='utf-8') as f:
    # Salvamos a nova lista (novo_dataset_limpo) em vez do dataset antigo
    json.dump(novo_dataset_limpo, f, ensure_ascii=False, indent=2)

print("Arquivo Dataset_clean.json gerado com sucesso no novo formato!")

ModuleNotFoundError: No module named 'ollama'